In [ ]:
# TPackages
import numpy as np
from google.colab import drive
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [ ]:
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Colab Notebooks/ro_puf_64bit_10000crps.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Quick reading of file
df=pd.read_csv(path)

In [ ]:
bit_cols = [f"c{i}" for i in range(64)]

def features(data, mode="ohe"):
    bits = data[bit_cols].values.astype(np.float32)

    if mode == "raw":
        return bits

    encoding_ro_i = np.eye(256, dtype=np.float32)[data["ro_i"].values.astype(int)]
    encoding_ro_j = np.eye(256, dtype=np.float32)[data["ro_j"].values.astype(int)]

    return np.concatenate([bits, encoding_ro_i, encoding_ro_j], axis=1).astype(np.float32)


In [ ]:
# Using Pytorch to create this model. We are testing how MLP is different from other models and using intervals of data to see just how effective the model is on the data
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

In [ ]:
# I am setting up the data to train. I want to have split the data up into training and validation
def train_model(data, mode="ohe", seed=42, epochs=25):
    y = data["response"].values.astype(np.float32)
    X = features(data, mode)

    X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=y)

    X_val, X_test, y_val, y_test = train_test_split(X_hold,y_hold,test_size=0.50,random_state=seed,stratify=y_hold)


    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

  # I chose the adam optimizer because it's usually the best performing one

  # I've tuned the LR multiple times. .003 was the best performing one
    model = MLP(X_train.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    X_train_go = torch.tensor(X_train)
    y_train_go = torch.tensor(y_train)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_go)
        loss = loss_fn(logits, y_train_go)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(torch.tensor(X_test))).numpy()

    preds = (probs >= 0.5).astype(int)

    return {
        "segment_rows": len(data),
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "features": "Raw 64-bit challenge" if mode == "raw" else "64 bits + OHE(ro_i, ro_j)",
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "auc": roc_auc_score(y_test, probs),
        "confusion_matrix": confusion_matrix(y_test, preds).tolist()
    }

In [ ]:
# I really want to see how the model performs at different intervals. My peers mentioned the model will be either really good or really bad. The best way to test it is by giving it differnt samples
# And see how it performs(to see if there's a drastic difference)

df_shuffled = df.sample(frac=1, random_state=7).reset_index(drop=True)
# My test ranges. Want to see how it'll work on 2k, 6k, and the whole dataset
results = []
for n in [2000, 6000, 10000]:
    segment = df_shuffled.iloc[:n].copy()
    for mode in ["raw", "ohe"]:
        results.append(train_model(segment, mode=mode, seed=42, epochs=25))

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("ro_puf_mlp_segment_results.csv", index=False)

,segment_rows,train_rows,test_rows,features,input_dim,accuracy,precision,recall,f1,auc,confusion_matrix
0,2000,1400,300,Raw 64-bit challenge,64,0.516667,0.520000,0.516556,0.518272,0.511178,"[[77, 72], [73, 78]]"
1,2000,1400,300,"64 bits + OHE(ro_i, ro_j)",576,0.753333,0.761905,0.741722,0.751678,0.832126,"[[114, 35], [39, 112]]"
2,6000,4200,900,Raw 64-bit challenge,64,0.536667,0.528474,0.524887,0.526674,0.555553,"[[251, 207], [210, 232]]"
3,6000,4200,900,"64 bits + OHE(ro_i, ro_j)",576,0.850000,0.869880,0.816742,0.842474,0.928941,"[[404, 54], [81, 361]]"
4,10000,7000,1500,Raw 64-bit challenge,64,0.522000,0.519444,0.502013,0.510580,0.544422,"[[409, 346], [371, 374]]"
5,10000,7000,1500,"64 bits + OHE(ro_i, ro_j)",576,0.818000,0.847059,0.773154,0.808421,0.925250,"[[651, 104], [169, 576]]"
